# ML-07 — Baseline Action Score and Top-10 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sujan-lab-cell/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This notebook implements the baseline action scoring rule and top-10 review for **Lane 2: Refresh / Content Opportunity Scoring**.

> Skill loaded: `building-baselines` + `flyrank/flyrank-data`.

## 1. My rule and its reason codes

### Plain Words Rule Definition
A webpage is prioritized for a content refresh if it is currently stuck in **striking distance (Google search position 11–20)**, has entered the **peak content decay age window (91–180 days since last update)**, and possesses sufficient organic impression volume to justify editorial intervention. Content in Top 3 positions is protected, while older content (>180 days) is deprioritized unless showing active position decay.

### Reason Codes & Action Labels
- `STRIKING_POSITION_PEAK_DECAY` → `REFRESH_AND_EXPAND_KEYWORDS`: High-leverage page in striking distance and peak decay window.
- `STRIKING_POSITION_OPPORTUNITY` → `OPTIMIZE_ON_PAGE_SEO`: Striking distance page needing title/heading optimization.
- `PEAK_DECAY_AGE_WINDOW` → `UPDATE_OUTDATED_SECTIONS`: Content aged 91–180 days experiencing freshness decline.
- `PAGE_1_TRAFFIC_PROTECTION` → `INTERNAL_LINK_BOOST`: Page 1 content (pos 4–10) needing internal link defense.
- `LOW_PRIORITY_STABLE` → `MONITOR_ONLY`: Top 3, deep rank, or fresh content requiring passive tracking.

--- 

### Signal 1 Audit (Flag-Linked Signal: Staleness)
FlyRank uses content staleness (`days_since_last_update` / `freshness_tier`) to trigger refresh flags. Below is the empirical bucket audit against the ground-truth outcome `is_declining_label` (`trend_direction == 'down'`).

**Verdict:** **MIXED**  
*Explanation:* Decline rate increases from **51.14%** (0–30d) to a peak of **61.11%** (91–180d). However, for content older than 180 days (`181+`), the decline rate drops sharply to **47.13%** (below the overall base rate of 54.21%). A simple linear heuristic assuming "older content is always more urgent" fails on >180d evergreen survivors. The 91–180d window represents the true peak decay zone.

--- 

### Signal 2 Audit (Rule-Leaning Signal: Position Tier)
We inspect performance decline across Google position tiers (`position_tier`).

**Verdict:** **CONFIRMED**  
*Explanation:* Content in striking position (avg position 11–20) exhibits the highest decline rate at **60.95%** (n=7,304), compared to only **24.08%** for Top 3 content and **34.42%** for Deep content. Striking distance content represents high decay vulnerability combined with maximum potential refresh leverage.

In [1]:
# Signal Bucket Audits Code Cell
import pandas as pd
import numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

base_rate = df['is_declining_label'].mean()
print(f"Overall Dataset Base Rate: {base_rate:.4f} ({base_rate*100:.2f}%)\n")

# Signal 1: Staleness (freshness_tier)
s1 = df.groupby('freshness_tier', observed=False)['is_declining_label'].agg(['count', 'mean']).rename(columns={'count': 'n', 'mean': 'decline_rate'})
s1['decline_rate_pct'] = (s1['decline_rate'] * 100).round(2)
print("=== SIGNAL 1 BUCKET TABLE: Staleness (freshness_tier) ===")
print(s1)
print("Verdict: MIXED (Peak decay at 91-180d, drops for 181+d)\n")

# Signal 2: Position Tier (position_tier)
s2 = df.groupby('position_tier', observed=False)['is_declining_label'].agg(['count', 'mean']).rename(columns={'count': 'n', 'mean': 'decline_rate'})
s2['decline_rate_pct'] = (s2['decline_rate'] * 100).round(2)
print("=== SIGNAL 2 BUCKET TABLE: Position Tier (position_tier) ===")
print(s2)
print("Verdict: CONFIRMED (Striking position 11-20 has highest decline rate at 60.95%)")

Overall Dataset Base Rate: 0.5421 (54.21%)

=== SIGNAL 1 BUCKET TABLE: Staleness (freshness_tier) ===
                    n  decline_rate  decline_rate_pct
freshness_tier                                       
0-30            20480      0.511377             51.14
181+              174      0.471264             47.13
31-90             175      0.588571             58.86
91-180           9171      0.611057             61.11
Verdict: MIXED (Peak decay at 91-180d, drops for 181+d)

=== SIGNAL 2 BUCKET TABLE: Position Tier (position_tier) ===
                   n  decline_rate  decline_rate_pct
position_tier                                       
deep            1319      0.344200             34.42
page_1         11814      0.569663             56.97
page_3_5        7242      0.561585             56.16
striking        7304      0.609529             60.95
top_3           2321      0.240844             24.08
Verdict: CONFIRMED (Striking position 11-20 has highest decline rate at 60.95%)


## 2. Build the ranked queue (writes the CSV)

We construct a transparent, unweighted multiplicative baseline scoring formula combining pre-period signals:

$$\text{baseline\_score} = (1.0 + 1.5 \cdot \mathbb{I}_{\text{striking}} + 0.8 \cdot \mathbb{I}_{\text{page\_1}}) \times (1.0 + 1.2 \cdot \mathbb{I}_{90 \le \text{age} \le 180} + 0.3 \cdot \mathbb{I}_{\text{age} > 180}) \times (1.0 + 0.5 \cdot \mathbb{I}_{\text{moderate\_imp}}) \times \ln(1 + \text{impressions\_90d})$$

This cell scores all 30,000 items, assigns reason codes and action labels, exports `work/outputs/baseline_action_score.csv`, and saves evaluation metrics to `work/outputs/baseline_metrics.json`.

In [2]:
# Build Ranked Queue and Write Outputs
import os
import json
import pandas as pd
import numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)
base_rate = df['is_declining_label'].mean()

# Inputs strictly knowable BEFORE prediction moment
is_striking = (df['position_tier'] == 'striking').astype(int)
is_page1 = (df['position_tier'] == 'page_1').astype(int)
is_peak_decay_age = ((df['days_since_last_update'] >= 90) & (df['days_since_last_update'] <= 180)).astype(int)
is_stale_180 = (df['days_since_last_update'] > 180).astype(int)
is_mod_imp = (df['impression_tier'] == 'moderate').astype(int)

df['baseline_score'] = (
    (1.0 + 1.5 * is_striking + 0.8 * is_page1) *
    (1.0 + 1.2 * is_peak_decay_age + 0.3 * is_stale_180) *
    (1.0 + 0.5 * is_mod_imp) *
    np.log1p(df['impressions_90d'])
).round(4)

# Assign ONE reason code per item
def get_reason_code(row):
    if row['position_tier'] == 'striking' and (90 <= row['days_since_last_update'] <= 180):
        return 'STRIKING_POSITION_PEAK_DECAY'
    elif row['position_tier'] == 'striking':
        return 'STRIKING_POSITION_OPPORTUNITY'
    elif (90 <= row['days_since_last_update'] <= 180):
        return 'PEAK_DECAY_AGE_WINDOW'
    elif row['position_tier'] == 'page_1':
        return 'PAGE_1_TRAFFIC_PROTECTION'
    else:
        return 'LOW_PRIORITY_STABLE'

# Assign ONE action label per item
def get_action_label(row):
    if row['position_tier'] == 'striking' and (90 <= row['days_since_last_update'] <= 180):
        return 'REFRESH_AND_EXPAND_KEYWORDS'
    elif row['position_tier'] == 'striking':
        return 'OPTIMIZE_ON_PAGE_SEO'
    elif (90 <= row['days_since_last_update'] <= 180):
        return 'UPDATE_OUTDATED_SECTIONS'
    elif row['position_tier'] == 'page_1':
        return 'INTERNAL_LINK_BOOST'
    else:
        return 'MONITOR_ONLY'

df['reason_code'] = df.apply(get_reason_code, axis=1)
df['action_label'] = df.apply(get_action_label, axis=1)

# Sort ranked queue
ranked_df = df.sort_values(by=['baseline_score', 'impressions_90d'], ascending=[False, False]).reset_index(drop=True)
ranked_df['rank'] = np.arange(1, len(ranked_df) + 1)

# Evaluate Precision@K
def precision_at_k(sorted_df, k):
    return float(sorted_df.head(k)['is_declining_label'].mean())

p10 = precision_at_k(ranked_df, 10)
p20 = precision_at_k(ranked_df, 20)
p50 = precision_at_k(ranked_df, 50)
p100 = precision_at_k(ranked_df, 100)
p500 = precision_at_k(ranked_df, 500)

print(f"=== BASELINE RULE EVALUATION ===")
print(f"Overall Base Rate: {base_rate:.4f} ({base_rate*100:.2f}%)")
print(f"Precision@10:  {p10:.4f} ({p10*100:.1f}%)")
print(f"Precision@20:  {p20:.4f} ({p20*100:.1f}%)")
print(f"Precision@50:  {p50:.4f} ({p50*100:.1f}%)")
print(f"Precision@100: {p100:.4f} ({p100*100:.1f}%)")
print(f"Precision@500: {p500:.4f} ({p500*100:.1f}%)\n")

# Export ranked CSV
os.makedirs("work/outputs", exist_ok=True)
output_cols = [
    'rank', 'content_id', 'client_id', 'baseline_score', 'reason_code', 'action_label',
    'impressions_90d', 'avg_position', 'position_tier', 'days_since_last_update', 'freshness_tier'
]
ranked_df[output_cols].to_csv("work/outputs/baseline_action_score.csv", index=False)
print(f"Successfully wrote ranked queue to work/outputs/baseline_action_score.csv ({len(ranked_df):,} rows)")

# Export metrics JSON
metrics = {
    "model_name": "rule_baseline_v1",
    "base_rate": base_rate,
    "precision_at_10": p10,
    "precision_at_20": p20,
    "precision_at_50": p50,
    "precision_at_100": p100,
    "precision_at_500": p500,
    "total_items": len(df)
}
with open("work/outputs/baseline_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print("Saved baseline metrics receipt to work/outputs/baseline_metrics.json")

=== BASELINE RULE EVALUATION ===
Overall Base Rate: 0.5421 (54.21%)
Precision@10:  0.6000 (60.0%)
Precision@20:  0.6000 (60.0%)
Precision@50:  0.7000 (70.0%)
Precision@100: 0.6300 (63.0%)
Precision@500: 0.6140 (61.4%)

Successfully wrote ranked queue to work/outputs/baseline_action_score.csv (30,000 rows)
Saved baseline metrics receipt to work/outputs/baseline_metrics.json


## 3. Top-20 review

Below is the line-by-line review of the **Top 10** items generated by our baseline rule, analyzed with a skeptic's eye to identify potential false positives.

| Rank | Content ID | Baseline Score | Action Label | Reason Code | Impressions (90d) | Avg Position | Days Unupdated | Label | What Would Make It Wrong? |
|---|---|---|---|---|---|---|---|---|---|
| 1 | `content_c5063073d048` | 66.91 | `REFRESH_AND_EXPAND_KEYWORDS` | `STRIKING_POSITION_PEAK_DECAY` | 192,205 | 12.5 | 104d | 0 | High impressions are driven by stable branded search terms where position 12.5 is stationary rather than decaying. |
| 2 | `content_eb366e871254` | 66.18 | `REFRESH_AND_EXPAND_KEYWORDS` | `STRIKING_POSITION_PEAK_DECAY` | 168,060 | 16.6 | 104d | 0 | Page targets broad intent queries with low conversion potential; updating text yields minimal organic traffic lift. |
| 3 | `content_4302c4925c0f` | 66.05 | `REFRESH_AND_EXPAND_KEYWORDS` | `STRIKING_POSITION_PEAK_DECAY` | 2,998 | 18.7 | 104d | 1 | Position 18.7 is caused by site-wide indexing/crawl budget constraints rather than outdated on-page content. |
| 4 | `content_51bb0bff5aed` | 66.04 | `REFRESH_AND_EXPAND_KEYWORDS` | `STRIKING_POSITION_PEAK_DECAY` | 2,993 | 13.0 | 104d | 1 | Competitors hold dominant backlink profiles on this keyword; content update alone cannot close the ranking gap. |
| 5 | `content_a0e08775e954` | 66.02 | `REFRESH_AND_EXPAND_KEYWORDS` | `STRIKING_POSITION_PEAK_DECAY` | 2,986 | 18.4 | 104d | 0 | Search intent shifted toward video/interactive tools, making text-based content refresh ineffective. |
| 6 | `content_a34aff7561ae` | 66.02 | `REFRESH_AND_EXPAND_KEYWORDS` | `STRIKING_POSITION_PEAK_DECAY` | 2,986 | 17.0 | 104d | 1 | Keyword search volume is highly seasonal and currently in an off-season dip; traffic will recover naturally. |
| 7 | `content_53b1fe682dee` | 65.99 | `REFRESH_AND_EXPAND_KEYWORDS` | `STRIKING_POSITION_PEAK_DECAY` | 2,977 | 15.9 | 104d | 1 | Page is already targeted by an active internal link campaign whose results have not yet propagated to Search Console. |
| 8 | `content_33cde508c590` | 65.98 | `REFRESH_AND_EXPAND_KEYWORDS` | `STRIKING_POSITION_PEAK_DECAY` | 2,973 | 11.0 | 104d | 1 | Position 11.0 is borderline Page 1; normal SERP turbulence can make stable traffic appear as temporary decline. |
| 9 | `content_a466b8ddcbbf` | 65.97 | `REFRESH_AND_EXPAND_KEYWORDS` | `STRIKING_POSITION_PEAK_DECAY` | 2,969 | 11.4 | 104d | 1 | Internal keyword cannibalization from a newly published page ranking at position 6; updating this page creates conflict. |
| 10 | `content_de79be03fa06` | 65.97 | `REFRESH_AND_EXPAND_KEYWORDS` | `STRIKING_POSITION_PEAK_DECAY` | 2,968 | 13.3 | 104d | 0 | Page content was recently modified on staging (metadata lag in `days_since_last_update` tracking). |

In [3]:
# Display Top 20 Ranked Queue DataFrame
top20_display = ranked_df.head(20)[['rank', 'content_id', 'client_id', 'baseline_score', 'reason_code', 'action_label', 'impressions_90d', 'avg_position', 'days_since_last_update', 'is_declining_label']]
print("=== TOP 20 RANKED QUEUE PREVIEW ===")
print(top20_display.to_string(index=False))

=== TOP 20 RANKED QUEUE PREVIEW ===
 rank           content_id         client_id  baseline_score                  reason_code                action_label  impressions_90d  avg_position  days_since_last_update  is_declining_label
    1 content_c5063073d048 client_6208ef0f77         66.9148 STRIKING_POSITION_PEAK_DECAY REFRESH_AND_EXPAND_KEYWORDS           192205          12.5                     104                   0
    2 content_eb366e871254 client_6208ef0f77         66.1765 STRIKING_POSITION_PEAK_DECAY REFRESH_AND_EXPAND_KEYWORDS           168060          16.6                     104                   0
    3 content_4302c4925c0f client_3fdba35f04         66.0498 STRIKING_POSITION_PEAK_DECAY REFRESH_AND_EXPAND_KEYWORDS             2998          18.7                     104                   1
    4 content_51bb0bff5aed client_6208ef0f77         66.0360 STRIKING_POSITION_PEAK_DECAY REFRESH_AND_EXPAND_KEYWORDS             2993          13.0                     104                   1

## 4. Weak picks + leakage check

### Weak Picks Analysis
In our top 20 review, **Rank 1 (`content_c5063073d048`)** and **Rank 2 (`content_eb366e871254`)** are weak picks (false positives). Because their 90-day impression counts (192,205 and 168,060) are massive outliers, the $\ln(1 + \text{impressions\_90d})$ factor dominated the rule score, pushing them to the top of the queue despite their `is_declining_label` being 0 (non-declining).  
This illustrates a fundamental limitation of fixed hand-written heuristics: linear or uncalibrated volume scaling over-prioritizes high-volume stable pages. This establishes the exact baseline benchmark that our Week-5 Machine Learning model must beat by learning complex, non-linear interactions without volume distortion.

--- 

### Target Leakage & Privacy Audit
We verify that zero target-derived columns (`trend_direction`, `trend_pct`, `is_declining_label`) or sub-window outcome features (`impressions_last_30d`, `clicks_last_30d`, `impressions_prev_30d`, `clicks_prev_30d`) entered the baseline score.

In [4]:
# Target Leakage Verification
excluded_cols = ['trend_direction', 'trend_pct', 'is_declining_label', 
                 'impressions_last_30d', 'clicks_last_30d', 'impressions_prev_30d', 'clicks_prev_30d']

rule_inputs = ['position_tier', 'days_since_last_update', 'impression_tier', 'impressions_90d']
leakage_overlap = set(rule_inputs).intersection(set(excluded_cols))

print("=== TARGET LEAKAGE AUDIT ===")
print(f"Baseline Rule Inputs: {rule_inputs}")
print(f"Forbidden Leakage Fields: {excluded_cols}")
print(f"Leakage Overlap Count: {len(leakage_overlap)}")
assert len(leakage_overlap) == 0, "ERROR: Target leakage detected in baseline rule!"
print("PASSED: Baseline rule inputs are 100% clean pre-period signals.")

=== TARGET LEAKAGE AUDIT ===
Baseline Rule Inputs: ['position_tier', 'days_since_last_update', 'impression_tier', 'impressions_90d']
Forbidden Leakage Fields: ['trend_direction', 'trend_pct', 'is_declining_label', 'impressions_last_30d', 'clicks_last_30d', 'impressions_prev_30d', 'clicks_prev_30d']
Leakage Overlap Count: 0
PASSED: Baseline rule inputs are 100% clean pre-period signals.


## Self-check

Before submitting, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.